# NASA Dataset Generation

This notebook is for creating a "combined" dataset. One only needs to download the data folder from Kaggle and run this notebook (with correct pathing.)

https://www.kaggle.com/datasets/vinayak123tyagi/bearing-dataset

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from tqdm import tqdm
from matplotlib import pyplot as plt
from scipy.spatial.distance import mahalanobis
sys.path.append("../")

In [ ]:
dataset2_path = "dataset_2/2nd_test/2nd_test"
#dataset3_path = "dataset_2/3rd_test/4th_test/txt" # not sure if we will work with the 3rd dataset. Have similar characteristics to dataset2. 

print("Changed working directory to:", os.getcwd())

In [38]:
file_list = sorted([f for f in os.listdir(dataset2_path)]) # store the file_name of each 10-minute file. 
df = [pd.read_csv(os.path.join(dataset2_path, file), delimiter='\t', header=None) for file in file_list] # joining the dataset2_path with each 10-min file. 

In [28]:
# check missing data

for i, j in enumerate(df):
    if j.isnull().values.any():
        print(f"Missing data in file {file_list[i]}")

# there doens't seem to be missing data here. 

In [ ]:
#creating one dataframe called "combined_data"

combined_data = pd.DataFrame()

for file in tqdm(file_list, desc='Processing Files'):
    file_path = os.path.join(dataset2_path, file)
    temp_df = pd.read_csv(file_path, delimiter='\t', header=None)
    temp_df['time'] = file # add metadata column
    combined_data = pd.concat([combined_data, temp_df], ignore_index=True)

In [ ]:
combined_data.index = range(len(combined_data)) # replace with index

combined_data.head()

In [ ]:
combined_data.columns = ['Bearing1', 'Bearing2', 'Bearing3', 'Bearing4', 'Time'] # assigning columns

combined_data.head()

In [ ]:
combined_data.to_csv('combined_dataset2.csv', index=False) # saving combined_dataset. 

# RMS Calculation

Below is the code for calculating the RMS values used in training my NN model. The original Kaggle data is organizaed in the following manner :

- one-second data is collected and every 10 mins, a file with its timestamp as the file name will be created. 

 The combined_dataset2.csv above contains one-second data from the Kaggle dataset set. The calculation below takes all the one-second data in each file and calculates a single RMS value for each 10 min file. 

In [ ]:
# # Group by the 'Time' column and calculate RMS for each bearing

df_rms = (
     combined_data.groupby('Time')
     .apply(lambda group: (group.iloc[:, :-1]**2).mean()**0.5))

df_rms = df_rms.reset_index()
df_rms.columns = ['Time'] + [f'RMS_Bearing_{i+1}'for i in range(df_rms.shape[1]-1)]

df_rms.head()


In [ ]:
# Plot RMS for each bearing

plt.figure(figsize=(10, 6))
for column in df_rms.columns[1:5]:  # Skip 'File' column
#     plt.plot( df_rms[column], label=column)
#     plt.ylabel('RMS')
#     plt.xlabel('index')

# plt.legend()

# plt.show()


In [ ]:
# rms_cols = [f'RMS_Bearing_{i}' for i in range(1,5)]
# bearing_data = df_rms[rms_cols].values

# baseline_data = bearing_data[:400]

# mean_bearing = np.mean(baseline_data, axis=0)
# cov_bearing = np.cov(baseline_data, rowvar=False)

# inv_cov_bearing = np.linalg.inv(cov_bearing)

# df_rms[f'MDistance'] = [mahalanobis(row, mean_bearing, inv_cov_bearing) for row in bearing_data]

# df_rms.head()

In [ ]:
# plt.plot(df_rms['MDistance'], label='Original Mahalanobis Distance')
# plt.xlabel('Time')
# plt.ylabel('Mahalanobis Distance')
# plt.legend()
# plt.show()

In [7]:
# threshold_percentile = 90
# threshold_value = np.percentile(df_rms['MDistance'], threshold_percentile)

# filtered_data = df_rms[df_rms['MDistance'] <= threshold_value]

In [ ]:
# healthy_mdistance = filtered_data['MDistance']#[0:600]
# print(healthy_mdistance)
# threshold = np.mean(healthy_mdistance)#10 # arbitrary threshold. 
# print(f'Threshold : {threshold}')

In [ ]:
# plt.plot( df_rms['MDistance'], label='MDistance')
# plt.axhline(y=threshold, color='r', linestyle='--', label='Threshold')
# plt.xlabel('Index')
# plt.ylabel('Mahalanobis Distance')
# plt.title("Full Data")
# plt.legend()
# plt.show()

In [ ]:
# df_rms.head()